# Feature Engineering — Historique des matchs et classement FIFA

## Objectif

Ce notebook prépare les variables qui serviront à prédire le résultat d'un match de football.

Le travail est organisé en deux grandes parties :

1. construire des statistiques à partir de l'historique des matchs ;
2. préparer ensuite le classement FIFA avant de le fusionner avec les matchs.

> Principe important : pour un match donné, nous utilisons uniquement les informations disponibles avant la date de ce match.


## 1. Importation de la bibliothèque

`pandas` permet de charger, examiner, nettoyer et transformer les données.


In [1]:
import pandas as pd

## 2. Chargement des données


In [2]:
matches = pd.read_csv("../data/raw/results.csv")
ranking = pd.read_csv("../data/raw/fifa_ranking-2024-06-20.csv")

## 3. Première exploration

Nous vérifions quelques lignes, les dimensions, les types et les valeurs manquantes.


In [3]:
print("Dimensions des matchs :", matches.shape)
display(matches.head())

Dimensions des matchs : (49499, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [4]:
print("Dimensions du classement FIFA :", ranking.shape)
display(ranking.head())

Dimensions du classement FIFA : (67472, 8)


,rank,country_full,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
0,140.0,Brunei Darussalam,BRU,2.0,0.0,140,AFC,1992-12-31
1,33.0,Portugal,POR,38.0,0.0,33,UEFA,1992-12-31
2,32.0,Zambia,ZAM,38.0,0.0,32,CAF,1992-12-31
3,31.0,Greece,GRE,38.0,0.0,31,UEFA,1992-12-31
4,30.0,Algeria,ALG,39.0,0.0,30,CAF,1992-12-31


In [5]:
matches.info()

<class 'pandas.DataFrame'>
RangeIndex: 49499 entries, 0 to 49498
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        49499 non-null  str    
 1   home_team   49499 non-null  str    
 2   away_team   49499 non-null  str    
 3   home_score  49490 non-null  float64
 4   away_score  49490 non-null  float64
 5   tournament  49499 non-null  str    
 6   city        49499 non-null  str    
 7   country     49499 non-null  str    
 8   neutral     49499 non-null  bool   
dtypes: bool(1), float64(2), str(6)
memory usage: 3.1 MB


In [6]:
ranking.info()

<class 'pandas.DataFrame'>
RangeIndex: 67472 entries, 0 to 67471
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   rank             67463 non-null  float64
 1   country_full     67472 non-null  str    
 2   country_abrv     67472 non-null  str    
 3   total_points     67472 non-null  float64
 4   previous_points  67472 non-null  float64
 5   rank_change      67472 non-null  int64  
 6   confederation    67472 non-null  str    
 7   rank_date        67472 non-null  str    
dtypes: float64(3), int64(1), str(4)
memory usage: 4.1 MB


In [7]:
print("Valeurs manquantes dans les matchs :")
display(matches.isna().sum())

print("Valeurs manquantes dans le classement FIFA :")
display(ranking.isna().sum())


Valeurs manquantes dans les matchs :


date          0
home_team     0
away_team     0
home_score    9
away_score    9
tournament    0
city          0
country       0
neutral       0
dtype: int64

Valeurs manquantes dans le classement FIFA :


rank               9
country_full       0
country_abrv       0
total_points       0
previous_points    0
rank_change        0
confederation      0
rank_date          0
dtype: int64

### Observation des matchs sans score

Un score manquant indique généralement qu'un match n'a pas encore été joué. Ces rencontres ne doivent pas servir au calcul de la forme passée.


In [8]:
matches[matches[["home_score", "away_score"]].isna().any(axis=1)]

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49490,2026-07-03,Australia,Egypt,NaN,NaN,FIFA World Cup,Arlington,United States,True
49491,2026-07-03,Argentina,Cape Verde,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True
49492,2026-07-03,Colombia,Ghana,NaN,NaN,FIFA World Cup,Kansas City,United States,True
49493,2026-07-04,Canada,Morocco,NaN,NaN,FIFA World Cup,Houston,United States,True
49494,2026-07-04,Paraguay,France,NaN,NaN,FIFA World Cup,Philadelphia,United States,True
49495,2026-07-05,Brazil,Norway,NaN,NaN,FIFA World Cup,East Rutherford,United States,True
49496,2026-07-05,Mexico,England,NaN,NaN,FIFA World Cup,Mexico City,Mexico,False
49497,2026-07-06,Portugal,Spain,NaN,NaN,FIFA World Cup,Dallas,United States,True
49498,2026-07-06,United States,Belgium,NaN,NaN,FIFA World Cup,Seattle,United States,False


### Observation des classements manquants

Nous observons les lignes concernées avant de décider de leur traitement dans la partie consacrée au classement FIFA.


In [9]:
ranking[ranking["rank"].isna()]

,rank,country_full,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
66339,NaN,Eritrea,ERI,855.56,0.0,0,CAF,2023-10-26
66340,NaN,Tonga,TGA,861.81,0.0,0,OFC,2023-10-26
66341,NaN,Samoa,SAM,894.26,0.0,0,OFC,2023-10-26
66342,NaN,American Samoa,ASA,900.27,0.0,0,OFC,2023-10-26
66493,NaN,Eritrea,ERI,855.56,0.0,0,CAF,2023-11-30
66734,NaN,Eritrea,ERI,855.56,0.0,0,CAF,2023-12-21
66919,NaN,Eritrea,ERI,855.56,0.0,0,CAF,2024-02-15
67134,NaN,Eritrea,ERI,855.56,0.0,0,CAF,2024-04-04
67393,NaN,Eritrea,ERI,855.56,0.0,0,CAF,2024-06-20


## 4. Préparation des dates


In [10]:
matches["date"] = pd.to_datetime(matches["date"])
ranking["rank_date"] = pd.to_datetime(ranking["rank_date"])

matches["year"] = matches["date"].dt.year

In [11]:
print("Période des matchs :", matches["date"].min(), "à", matches["date"].max())
print("Période du classement :", ranking["rank_date"].min(), "à", ranking["rank_date"].max())

Période des matchs : 1872-11-30 00:00:00 à 2026-07-06 00:00:00
Période du classement : 1992-12-31 00:00:00 à 2024-06-20 00:00:00


## 5. Conservation des matchs déjà joués

Nous retirons uniquement les rencontres dont le score est absent. Nous créons ensuite un identifiant unique pour chaque match.


In [12]:
matches_played = (
    matches
    .dropna(subset=["home_score", "away_score"])
    .reset_index(drop=True)
    .copy()
)

matches_played["match_id"] = matches_played.index


In [13]:
print("Nombre de matchs joués :", len(matches_played))
print("Nombre d'identifiants uniques :", matches_played["match_id"].nunique())

assert matches_played["match_id"].is_unique


Nombre de matchs joués : 49490
Nombre d'identifiants uniques : 49490


## 6. Transformation au format équipe-match

Un match contient deux équipes. Nous créons donc deux lignes par match :

- une ligne du point de vue de l'équipe à domicile ;
- une ligne du point de vue de l'équipe à l'extérieur.

Ce format facilite le calcul de l'historique propre à chaque équipe.


In [14]:
def get_team_stats(match, team, is_home):
    if is_home:
        goals_for = match["home_score"]
        goals_against = match["away_score"]
        opponent = match["away_team"]
    else:
        goals_for = match["away_score"]
        goals_against = match["home_score"]
        opponent = match["home_team"]

    return {
        "match_id": match["match_id"],
        "date": match["date"],
        "team": team,
        "opponent": opponent,
        "is_home": is_home,
        "goals_for": goals_for,
        "goals_against": goals_against,
        "win": int(goals_for > goals_against),
        "draw": int(goals_for == goals_against),
        "loss": int(goals_for < goals_against),
    }


In [15]:
team_rows = []

for _, match in matches_played.iterrows():
    team_rows.append(
        get_team_stats(match, match["home_team"], is_home=True)
    )
    team_rows.append(
        get_team_stats(match, match["away_team"], is_home=False)
    )

team_matches = pd.DataFrame(team_rows)


In [16]:
print("Nombre de matchs :", len(matches_played))
print("Nombre de lignes équipe-match :", len(team_matches))
display(team_matches.head())

assert len(team_matches) == 2 * len(matches_played)


Nombre de matchs : 49490
Nombre de lignes équipe-match : 98980


,match_id,date,team,opponent,is_home,goals_for,goals_against,win,draw,loss
0,0,1872-11-30,Scotland,England,True,0.0,0.0,0,1,0
1,0,1872-11-30,England,Scotland,False,0.0,0.0,0,1,0
2,1,1873-03-08,England,Scotland,True,4.0,2.0,1,0,0
3,1,1873-03-08,Scotland,England,False,2.0,4.0,0,0,1
4,2,1874-03-07,Scotland,England,True,2.0,1.0,1,0,0


## 7. Calcul de la forme sur les 5 matchs précédents

Les données sont triées par équipe et par date. `shift(1)` décale les résultats d'une ligne : le match actuel n'entre donc jamais dans ses propres variables explicatives.


In [17]:
team_matches = (
    team_matches
    .sort_values(["team", "date", "match_id"])
    .reset_index(drop=True)
)


In [18]:
team_matches["recent_wins"] = (
    team_matches.groupby("team")["win"]
    .transform(lambda s: s.shift(1).rolling(5, min_periods=1).sum())
)

team_matches["recent_draws"] = (
    team_matches.groupby("team")["draw"]
    .transform(lambda s: s.shift(1).rolling(5, min_periods=1).sum())
)

team_matches["recent_losses"] = (
    team_matches.groupby("team")["loss"]
    .transform(lambda s: s.shift(1).rolling(5, min_periods=1).sum())
)

team_matches["avg_goals_for"] = (
    team_matches.groupby("team")["goals_for"]
    .transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
)

team_matches["avg_goals_against"] = (
    team_matches.groupby("team")["goals_against"]
    .transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
)

team_matches["recent_matches_count"] = (
    team_matches.groupby("team")["win"]
    .transform(lambda s: s.shift(1).rolling(5, min_periods=1).count())
)


### Traitement des premières rencontres

Lorsqu'une équipe n'a aucun match antérieur, ses statistiques récentes sont absentes. Nous les mettons à zéro. La colonne `recent_matches_count` permet au modèle de reconnaître qu'il s'agit d'une absence d'historique.


In [19]:
recent_columns = [
    "recent_matches_count",
    "recent_wins",
    "recent_draws",
    "recent_losses",
    "avg_goals_for",
    "avg_goals_against",
]

team_matches[recent_columns] = team_matches[recent_columns].fillna(0)


In [20]:
display(
    team_matches.loc[team_matches["team"] == "France", [
        "date", "opponent", "goals_for", "goals_against",
        "recent_matches_count", "recent_wins", "recent_draws",
        "recent_losses", "avg_goals_for", "avg_goals_against"
    ]].tail(10)
)


,date,opponent,goals_for,goals_against,recent_matches_count,recent_wins,recent_draws,recent_losses,avg_goals_for,avg_goals_against
29823,2025-11-13,Ukraine,4.0,0.0,5.0,4.0,1.0,0.0,2.2,0.6
29824,2025-11-16,Azerbaijan,3.0,1.0,5.0,4.0,1.0,0.0,2.6,0.6
29825,2026-03-26,Brazil,2.0,1.0,5.0,4.0,1.0,0.0,2.8,0.8
29826,2026-03-29,Colombia,3.0,1.0,5.0,4.0,1.0,0.0,2.8,0.8
29827,2026-06-04,Ivory Coast,1.0,2.0,5.0,4.0,1.0,0.0,2.8,1.0
29828,2026-06-08,Northern Ireland,3.0,1.0,5.0,4.0,0.0,1.0,2.6,1.0
29829,2026-06-16,Senegal,3.0,1.0,5.0,4.0,0.0,1.0,2.4,1.2
29830,2026-06-22,Iraq,3.0,0.0,5.0,4.0,0.0,1.0,2.4,1.2
29831,2026-06-26,Norway,4.0,1.0,5.0,4.0,0.0,1.0,2.6,1.0
29832,2026-06-30,Sweden,3.0,0.0,5.0,4.0,0.0,1.0,2.8,1.0


## 8. Séparation des statistiques domicile et extérieur

Nous sélectionnons exactement une ligne domicile et une ligne extérieur pour chaque `match_id`. Cette méthode évite les doublons causés par une fusion sur la date et les noms des équipes.


In [21]:
home_form = team_matches.loc[
    team_matches["is_home"],
    ["match_id"] + recent_columns
].copy()

home_form = home_form.rename(columns={
    column: f"home_{column}" for column in recent_columns
})

away_form = team_matches.loc[
    ~team_matches["is_home"],
    ["match_id"] + recent_columns
].copy()

away_form = away_form.rename(columns={
    column: f"away_{column}" for column in recent_columns
})

In [22]:
print("Lignes domicile :", len(home_form))
print("Lignes extérieur :", len(away_form))

assert home_form["match_id"].is_unique
assert away_form["match_id"].is_unique
assert len(home_form) == len(matches_played)
assert len(away_form) == len(matches_played)


Lignes domicile : 49490
Lignes extérieur : 49490


## 9. Fusion sécurisée avec la table des matchs

La fusion utilise uniquement `match_id`, qui est une clé unique. `validate="one_to_one"` demande à pandas de bloquer l'opération si une table contient plusieurs lignes pour le même match.


In [23]:
matches_features = matches_played.merge(
    home_form,
    on="match_id",
    how="left",
    validate="one_to_one",
)

matches_features = matches_features.merge(
    away_form,
    on="match_id",
    how="left",
    validate="one_to_one",
)


In [24]:
print("Matchs avant fusion :", matches_played.shape)
print("Matchs après fusion :", matches_features.shape)

assert len(matches_features) == len(matches_played)
assert matches_features["match_id"].is_unique


Matchs avant fusion : (49490, 11)
Matchs après fusion : (49490, 23)


## 10. Création des différences entre les deux équipes

Une différence positive signifie généralement que l'équipe à domicile possède une valeur plus élevée que l'équipe extérieure.


In [25]:
matches_features["recent_wins_diff"] = (
    matches_features["home_recent_wins"]
    - matches_features["away_recent_wins"]
)

matches_features["recent_draws_diff"] = (
    matches_features["home_recent_draws"]
    - matches_features["away_recent_draws"]
)

matches_features["recent_losses_diff"] = (
    matches_features["home_recent_losses"]
    - matches_features["away_recent_losses"]
)

matches_features["avg_goals_for_diff"] = (
    matches_features["home_avg_goals_for"]
    - matches_features["away_avg_goals_for"]
)

matches_features["avg_goals_against_diff"] = (
    matches_features["home_avg_goals_against"]
    - matches_features["away_avg_goals_against"]
)

matches_features["home_recent_form_points"] = (
    3 * matches_features["home_recent_wins"]
    + matches_features["home_recent_draws"]
)

matches_features["away_recent_form_points"] = (
    3 * matches_features["away_recent_wins"]
    + matches_features["away_recent_draws"]
)

matches_features["recent_form_points_diff"] = (
    matches_features["home_recent_form_points"]
    - matches_features["away_recent_form_points"]
)


## 11. Contrôle final de cette première partie


In [26]:
feature_columns = [
    column for column in matches_features.columns
    if column.startswith("home_")
    or column.startswith("away_")
    or column.endswith("_diff")
]

print("Dimensions finales :", matches_features.shape)
print("Valeurs manquantes dans les nouvelles variables :")
display(matches_features[feature_columns].isna().sum())

display(matches_features.head())


Dimensions finales : (49490, 31)
Valeurs manquantes dans les nouvelles variables :


home_team                    0
away_team                    0
home_score                   0
away_score                   0
home_recent_matches_count    0
home_recent_wins             0
home_recent_draws            0
home_recent_losses           0
home_avg_goals_for           0
home_avg_goals_against       0
away_recent_matches_count    0
away_recent_wins             0
away_recent_draws            0
away_recent_losses           0
away_avg_goals_for           0
away_avg_goals_against       0
recent_wins_diff             0
recent_draws_diff            0
recent_losses_diff           0
avg_goals_for_diff           0
avg_goals_against_diff       0
home_recent_form_points      0
away_recent_form_points      0
recent_form_points_diff      0
dtype: int64

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,...,away_avg_goals_for,away_avg_goals_against,recent_wins_diff,recent_draws_diff,recent_losses_diff,avg_goals_for_diff,avg_goals_against_diff,home_recent_form_points,away_recent_form_points,recent_form_points_diff
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,1872,...,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,1873,...,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,1.0,1.0,0.0
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,1874,...,2.000000,1.000000,-1.0,0.0,1.0,-1.000000,1.000000,1.0,4.0,-3.0
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,1875,...,1.333333,1.666667,0.0,0.0,0.0,0.333333,-0.333333,4.0,4.0,0.0
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,1876,...,1.750000,1.500000,0.0,0.0,0.0,-0.250000,0.250000,5.0,5.0,0.0


## 12. Préparation du classement FIFA

Le nom harmonisé de l'équipe sert d'identifiant, car les deux sources ne possèdent pas d'ID numérique commun. Pour chaque match, nous récupérons le dernier classement publié à cette date ou avant cette date avec une jointure temporelle.

> `direction="backward"` empêchera d'utiliser un classement publié après le match. Les sélections non affiliées à la FIFA resteront sans classement.


In [27]:
ranking_clean = ranking.copy()
assert not ranking_clean.duplicated(["country_full", "rank_date"]).any()

team_name_mapping = {
    "Brunei": "Brunei Darussalam", "Cape Verde": "Cabo Verde",
    "China": "China PR", "Curaçao": "Curacao",
    "Czech Republic": "Czechia", "DR Congo": "Congo DR",
    "Gambia": "The Gambia", "Iran": "IR Iran",
    "Ivory Coast": "Côte d'Ivoire", "Kyrgyzstan": "Kyrgyz Republic",
    "North Korea": "Korea DPR", "South Korea": "Korea Republic",
    "Saint Kitts and Nevis": "St Kitts and Nevis",
    "Saint Lucia": "St Lucia",
    "Saint Vincent and the Grenadines": "St Vincent and the Grenadines",
    "São Tomé and Príncipe": "Sao Tome and Principe",
    "Taiwan": "Chinese Taipei", "United States": "USA",
    "United States Virgin Islands": "US Virgin Islands",
}
matches_features["home_team_key"] = matches_features["home_team"].replace(team_name_mapping)
matches_features["away_team_key"] = matches_features["away_team"].replace(team_name_mapping)
ranking_clean["team_key"] = ranking_clean["country_full"]


In [28]:
def add_fifa_ranking(matches_table, side):
    team_column = f"{side}_team_key"
    fifa = ranking_clean[["team_key", "rank_date", "rank", "total_points", "rank_change", "confederation"]].rename(columns={
        "team_key": team_column,
        **{c: f"{side}_{c}" for c in ["rank_date", "rank", "total_points", "rank_change", "confederation"]}
    })
    return pd.merge_asof(
        matches_table.sort_values(["date", team_column]),
        fifa.sort_values([f"{side}_rank_date", team_column]),
        left_on="date", right_on=f"{side}_rank_date", by=team_column,
        direction="backward", allow_exact_matches=True,
    )

matches_with_fifa = add_fifa_ranking(matches_features, "home")
matches_with_fifa = add_fifa_ranking(matches_with_fifa, "away")
matches_with_fifa = matches_with_fifa.sort_values("match_id").reset_index(drop=True)

assert len(matches_with_fifa) == len(matches_features)
assert matches_with_fifa["match_id"].is_unique
for side in ["home", "away"]:
    known = matches_with_fifa[f"{side}_rank_date"].notna()
    assert (matches_with_fifa.loc[known, f"{side}_rank_date"] <= matches_with_fifa.loc[known, "date"]).all()


## 13. Variables FIFA et variable cible

Un rang plus petit est meilleur : `away_rank - home_rank` est donc positif lorsque l'équipe à domicile est mieux classée. Les scores servent uniquement à construire la cible et seront exclus des variables du modèle.


In [29]:
matches_with_fifa["rank_diff"] = matches_with_fifa["away_rank"] - matches_with_fifa["home_rank"]
matches_with_fifa["total_points_diff"] = matches_with_fifa["home_total_points"] - matches_with_fifa["away_total_points"]
for side in ["home", "away"]:
    matches_with_fifa[f"{side}_ranking_age_days"] = (matches_with_fifa["date"] - matches_with_fifa[f"{side}_rank_date"]).dt.days

matches_with_fifa["target"] = 1  # match nul
matches_with_fifa.loc[matches_with_fifa["home_score"] > matches_with_fifa["away_score"], "target"] = 0  # victoire domicile
matches_with_fifa.loc[matches_with_fifa["home_score"] < matches_with_fifa["away_score"], "target"] = 2  # victoire extérieur

eligible_for_fifa = matches_with_fifa["date"] >= ranking_clean["rank_date"].min()
modeling_data = matches_with_fifa.loc[eligible_for_fifa].sort_values(["date", "match_id"]).reset_index(drop=True).copy()
print("Dimensions du jeu candidat :", modeling_data.shape)
print("Couverture des rangs domicile/extérieur :")
display(modeling_data[["home_rank", "away_rank"]].notna().mean())
display(modeling_data["target"].value_counts(normalize=True).round(3))


Dimensions du jeu candidat : (30782, 48)
Couverture des rangs domicile/extérieur :


home_rank    0.949938
away_rank    0.945618
dtype: float64

target
home_win    0.486
away_win    0.281
draw        0.233
Name: proportion, dtype: float64

## 14. Export du jeu préparé

Le jeu fusionné est enregistré séparément des données brutes pour rendre la modélisation indépendante et reproductible.


In [ ]:
from pathlib import Path
processed_directory = Path("../data/processed")
processed_directory.mkdir(parents=True, exist_ok=True)
modeling_data.to_csv(processed_directory / "modeling_data.csv", index=False)
print("Jeu préparé enregistré :", processed_directory / "modeling_data.csv")
